# 2.0 EDA via Athena (run in SageMaker Studio)

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import boto3
from src.pamap2.config import load_config, s3_uri
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
sts = session.client("sts")
print("Caller:", sts.get_caller_identity()["Arn"])
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src.pamap2 import schema
import io, time, pandas as pd
athena = session.client("athena")
s3 = session.client("s3")
def run(sql, timeout=600):
    qid = athena.start_query_execution(QueryString=sql, WorkGroup=cfg["athena_workgroup"],
        QueryExecutionContext={"Database": cfg["glue_database"]})["QueryExecutionId"]
    deadline = time.time() + timeout
    while True:
        qe = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        st = qe["Status"]
        if st["State"] in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        if time.time() > deadline:
            athena.stop_query_execution(QueryExecutionId=qid)
            raise TimeoutError(f"Athena query exceeded {timeout}s: {sql[:200]}")
        time.sleep(1)
    if st["State"] != "SUCCEEDED":
        raise RuntimeError(f"{st['State']}: {st.get('StateChangeReason')}\n{sql}")
    if qe.get("StatementType") != "DML":
        return None
    out = qe["ResultConfiguration"]["OutputLocation"]
    b, k = out[len("s3://"):].split("/", 1)
    return pd.read_csv(io.BytesIO(s3.get_object(Bucket=b, Key=k)["Body"].read()))
q = run
T = cfg['curated_table']

## Class distribution

In [ ]:
print("Transient activity 0 rows (excluded below):", q(f"SELECT COUNT(*) AS n FROM {T} WHERE activity_id = 0").n[0])
cls = q(f"SELECT activity_id, COUNT(*) AS n FROM {T} WHERE activity_id > 0 GROUP BY 1 ORDER BY 1")
cls["activity"] = cls.activity_id.map(schema.ACTIVITY_NAMES)
cls["pct"] = 100 * cls.n / cls.n.sum()
ax = cls.set_index("activity").n.plot.barh(figsize=(8, 6)); ax.set_title("Rows per activity"); plt.show()
cls

## Per-subject counts and activity coverage

In [ ]:
subj = q(f"SELECT subject_id, session, COUNT(*) AS n, COUNT(DISTINCT activity_id) AS n_activities FROM {T} GROUP BY 1,2 ORDER BY 1,2")
print(subj, "\nTotal rows:", subj.n.sum())
sa = q(f"SELECT subject_id, activity_id, COUNT(*) AS n FROM {T} GROUP BY 1,2").pivot(index="subject_id", columns="activity_id", values="n").fillna(0)
sns.heatmap(sa, cmap="viridis"); plt.title("Rows per subject x activity"); plt.show()

## Missingness (heart rate is sparse, ~9 Hz vs 100 Hz)

In [ ]:
cols = [c for c in schema.CURATED_COLUMNS if c not in ("timestamp", "activity_id")]
sel = ", ".join(f"AVG(CASE WHEN {c} IS NULL THEN 1.0 ELSE 0.0 END) AS {c}" for c in cols)
miss = q(f"SELECT {sel} FROM {T}").iloc[0].astype(float).sort_values(ascending=False) * 100
miss.head(15).plot.barh(figsize=(8, 5)); plt.xlabel("% missing"); plt.title("Top missing columns"); plt.show()
q(f"SELECT subject_id, 100*AVG(CASE WHEN heart_rate IS NULL THEN 1.0 ELSE 0.0 END) AS hr_missing_pct FROM {T} GROUP BY 1 ORDER BY 1")

## Sensor distributions (1% sample drawn in Athena)

In [ ]:
sample_cols = ["activity_id", "heart_rate", "hand_acc16_x", "hand_acc16_y", "hand_acc16_z", "chest_acc16_x",
               "chest_acc16_y", "chest_acc16_z", "ankle_acc16_x", "ankle_acc16_y", "ankle_acc16_z",
               "hand_gyro_x", "chest_gyro_x", "ankle_gyro_x"]
smp = q(f"SELECT {', '.join(sample_cols)} FROM {T} WHERE activity_id > 0 AND rand() < 0.01")
smp.drop(columns="activity_id").hist(figsize=(14, 9), bins=60); plt.tight_layout(); plt.show()

## Correlations

In [ ]:
plt.figure(figsize=(10, 8)); sns.heatmap(smp.drop(columns="activity_id").corr(), cmap="coolwarm", center=0); plt.title("Sensor correlations"); plt.show()

## Activity durations (100 Hz)

In [ ]:
dur = q(f"SELECT subject_id, activity_id, COUNT(*)/100.0 AS seconds FROM {T} WHERE activity_id > 0 GROUP BY 1,2")
dur["activity"] = dur.activity_id.map(schema.ACTIVITY_NAMES)
plt.figure(figsize=(10, 6)); sns.boxplot(data=dur, y="activity", x="seconds"); plt.title("Seconds per subject, by activity"); plt.show()

## Time series per activity (subject 101, protocol, chest accelerometer)

In [ ]:
ts = q(f"SELECT \"timestamp\", activity_id, chest_acc16_x, chest_acc16_y, chest_acc16_z FROM {T} WHERE subject_id = 101 AND session = 'protocol' ORDER BY \"timestamp\"")
acts = [a for a in ts.activity_id.unique() if a > 0]
if not acts:
    print("No data for subject 101 / protocol")
else:
  fig, axes = plt.subplots(len(acts), 1, figsize=(12, 2 * len(acts)))
  for ax, a in zip(np.atleast_1d(axes), acts):
    seg = ts[ts.activity_id == a].head(1000)
    ax.plot(seg.timestamp, seg[["chest_acc16_x", "chest_acc16_y", "chest_acc16_z"]]); ax.set_ylabel(schema.ACTIVITY_NAMES[a], rotation=0, ha="right")
  plt.tight_layout(); plt.show()

## Findings (feeds spec B)

Confirm against the plots above after running:

- **Class imbalance:** activity 0 (transient/other) is the largest class and must be removed or relabeled; several activities (e.g. 9, 10, 11, 12, 13, 16-20, 24) are rare or absent for some subjects (see the subject x activity heatmap), so subject-aware splits must handle missing classes.
- **Heart rate missingness:** HR is far sparser than IMU columns (~9 Hz vs 100 Hz); forward-fill/interpolate in spec B per subject and session, never across subjects.
- **Sensor scale:** acceleration, gyroscope and magnetometer ranges differ by orders of magnitude and by placement; standardize per channel, fit on train only.
- **Redundancy:** acc16 and acc6 channels are highly correlated; consider keeping only acc16.
- **Durations:** activity durations vary widely per subject, so windows per class are uneven; use class weights or resampling.
- **Subject variability:** per-subject distributions differ, motivating subject-aware (not row-random) splitting.